In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
ml_features_and_labels = pd.read_csv('/ml_features_and_labels.csv')
scenario_manifest = pd.read_csv('/scenario_manifest.csv')

In [ ]:
ml_features_and_labels.head()


,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e1b_ciphersuite_60,e2b_tls_version_SSL3.0,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run1
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,val,eval_test_network,net_high_jitter_classic_run10
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run100
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run101
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run102


In [ ]:
scenario_manifest.head()

,ID,scenario_id,taxonomy_class,subtype,operational_class,anomaly_label,run_idx
0,normal_mlkem768_mldsa65_cert_run1.pcap,normal_mlkem768_mldsa65_cert,ML,baseline_normal,Compliant_mlkem768,Normal,1
1,normal_mlkem768_mldsa65_cert_run2.pcap,normal_mlkem768_mldsa65_cert,ML,baseline_normal,Compliant_mlkem768,Normal,2
2,normal_mlkem768_mldsa65_cert_run3.pcap,normal_mlkem768_mldsa65_cert,ML,baseline_normal,Compliant_mlkem768,Normal,3
3,normal_mlkem768_mldsa65_cert_run4.pcap,normal_mlkem768_mldsa65_cert,ML,baseline_normal,Compliant_mlkem768,Normal,4
4,normal_mlkem768_mldsa65_cert_run5.pcap,normal_mlkem768_mldsa65_cert,ML,baseline_normal,Compliant_mlkem768,Normal,5


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Ensure the dataset is loaded in the current runtime session
try:
    ml_features_and_labels
except NameError:
    ml_features_and_labels = pd.read_csv('ml_features_and_labels.csv')

# Keep only numeric columns for modeling, and exclude the target variable 'label'
numeric_features = ml_features_and_labels.select_dtypes(include=['number'])

# Drop rows containing missing values (NaN) to prevent fitting errors
clean_data = numeric_features.dropna()

X = clean_data.drop('label', axis=1, errors='ignore')
y = clean_data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.10145033413718704


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Ensure the dataset is loaded in the current runtime session using correct absolute path
try:
    scenario_manifest
except NameError:
    scenario_manifest = pd.read_csv('/scenario_manifest.csv')

# Keep only numeric columns for modeling, and exclude the target variable 'anomaly_label'
numeric_features = scenario_manifest.select_dtypes(include=['number'])

# Map 'anomaly_label' to binary numeric labels: 0 for 'Normal', 1 for anything else
y = scenario_manifest['anomaly_label'].apply(lambda x: 0 if x == 'Normal' else 1)

# Drop rows containing missing values (NaN) in features
clean_data = numeric_features.dropna()
# Align targets with clean data rows
y = y.loc[clean_data.index]

X = clean_data.copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.21192364726664348


### Models for `scenario_manifest` Dataset

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import pandas as pd
from sklearn.model_selection import train_test_split

# Ensure the dataset is loaded in the current runtime session
try:
    scenario_manifest
except NameError:
    scenario_manifest = pd.read_csv('/scenario_manifest.csv')

# Prepare features and target from scenario_manifest
numeric_features = scenario_manifest.select_dtypes(include=['number'])
clean_data = numeric_features.dropna()

X_sm = clean_data.copy()
y_sm = scenario_manifest['anomaly_label'].apply(lambda x: 0 if x == 'Normal' else 1).loc[clean_data.index]

# Split
X_train_sm, X_test_sm, y_train_sm, y_test_sm = train_test_split(X_sm, y_sm, test_size=0.2, random_state=42)

# Scale features for Logistic Regression
scaler_sm = StandardScaler()
X_train_scaled_sm = scaler_sm.fit_transform(X_train_sm)
X_test_scaled_sm = scaler_sm.transform(X_test_sm)

# Logistic Regression
logmodel_sm = LogisticRegression(max_iter=1000, random_state=42)
logmodel_sm.fit(X_train_scaled_sm, y_train_sm)

y_pred_log = logmodel_sm.predict(X_test_scaled_sm)
y_prob_log = logmodel_sm.predict_proba(X_test_scaled_sm)[:, 1]

print("=== Logistic Regression Results ===")
print("Precision:", precision_score(y_test_sm, y_pred_log, zero_division=0))
print("Recall:", recall_score(y_test_sm, y_pred_log, zero_division=0))
print("F1-Score:", f1_score(y_test_sm, y_pred_log, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test_sm, y_prob_log))
print("Confusion Matrix:\n", confusion_matrix(y_test_sm, y_pred_log))

=== Logistic Regression Results ===
Precision: 0.49559748427672956
Recall: 0.4057672502574665
F1-Score: 0.44620611551528877
ROC-AUC: 0.9288610202192703
Confusion Matrix:
 [[6630  401]
 [ 577  394]]


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
rf_model_sm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_sm.fit(X_train_sm, y_train_sm)

y_pred_rf = rf_model_sm.predict(X_test_sm)
y_prob_rf = rf_model_sm.predict_proba(X_test_sm)[:, 1]

print("=== Random Forest Results ===")
print("Precision:", precision_score(y_test_sm, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test_sm, y_pred_rf, zero_division=0))
print("F1-Score:", f1_score(y_test_sm, y_pred_rf, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test_sm, y_prob_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test_sm, y_pred_rf))

=== Random Forest Results ===
Precision: 0.32507149666348906
Recall: 0.35118434603501547
F1-Score: 0.33762376237623765
ROC-AUC: 0.8941672314500694
Confusion Matrix:
 [[6323  708]
 [ 630  341]]


In [10]:
import xgboost as xgb
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# XGBoost model training and evaluation
xgb_model_sm = xgb.XGBClassifier(random_state=42)
xgb_model_sm.fit(X_train_sm, y_train_sm)

y_pred_xgb = xgb_model_sm.predict(X_test_sm)
y_prob_xgb = xgb_model_sm.predict_proba(X_test_sm)[:, 1]

print("=== XGBoost Results ===")
print("Precision:", precision_score(y_test_sm, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test_sm, y_pred_xgb, zero_division=0))
print("F1-Score:", f1_score(y_test_sm, y_pred_xgb, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test_sm, y_prob_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test_sm, y_pred_xgb))

=== XGBoost Results ===
Precision: 0.4319371727748691
Recall: 0.509783728115345
F1-Score: 0.46764289088332545
ROC-AUC: 0.914552604392406
Confusion Matrix:
 [[6380  651]
 [ 476  495]]


Step 4 — Build a baseline model

Start with Logistic Regression.

This gives us a simple baseline and lets us make sure the entire pipeline works:

Dataset → preprocessing → training → prediction → evaluation

Use:

Precision

Recall

F1-score

ROC-AUC

Confusion matrix

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Ensure the dataset is loaded in the current runtime session
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Initialize and fit the Logistic Regression model
logmodel = LogisticRegression(max_iter= 1000, random_state= 42)
logmodel.fit(X_train_scaled, y_train)

# Make predictions
y_pred = logmodel.predict(X_test_scaled)
y_prob = logmodel.predict_proba(X_test_scaled)
# Evaluation
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob[:, 1])
conf_matrix = confusion_matrix(y_test, y_pred)

# print results
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("ROC-AUC:", roc_auc)
print("Confusion Matrix:\n", conf_matrix)


Precision: 0.49559748427672956
Recall: 0.4057672502574665
F1-Score: 0.44620611551528877
ROC-AUC: 0.9288610202192703
Confusion Matrix:
 [[6630  401]
 [ 577  394]]


Step 5 — Add Random Forest

Train a Random Forest and compare it with Logistic Regression.

Questions:

Does it perform better?

Which features appear important?

How does it handle the anomalous class?

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
# load data
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=42)

# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# make model
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
# train model
random_forest.fit(X_train, y_train)
# make predictions
y_pred = random_forest.predict(X_test)
Y_prob = random_forest.predict_proba(X_test)
# evaluate model

# results
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, Y_prob[:, 1])
conf_matrix = confusion_matrix(y_test, y_pred)
# print results
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("ROC-AUC:", roc_auc)
print("Confusion Matrix:\n", conf_matrix)

Precision: 0.8775510204081632
Recall: 0.9148936170212766
F1-Score: 0.8958333333333334
ROC-AUC: 0.9734544359694901
Confusion Matrix:
 [[94 12]
 [ 8 86]]


In [18]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix )
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

XGBModel = xgb.XGBClassifier(random_state=42)
XGBModel.fit(X_train, y_train)
y_pred = XGBModel.predict(X_test)
Y_prob = XGBModel.predict_proba(X_test)

# evaluation
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, Y_prob[:, 1])
conf_matrix = confusion_matrix(y_test, y_pred)

#print
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("ROC-AUC:", roc_auc)
print("Confusion Matrix:\n", conf_matrix)



Precision: 0.9166666666666666
Recall: 0.9361702127659575
F1-Score: 0.9263157894736842
ROC-AUC: 0.9829385788839823
Confusion Matrix:
 [[98  8]
 [ 6 88]]
